In [ ]:
pip install pandas sqlalchemy pymysql


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Loading the store data set
stores_df = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Stores.csv", encoding='ISO-8859-1')

stores_df.isnull().sum()

In [ ]:
# Fill with the mean value
stores_df['Square Meters'].fillna(stores_df['Square Meters'].mean(), inplace=True)
stores_df.isnull().sum()

In [ ]:
stores_df.dtypes

In [ ]:
# Convert 'Open Date' to datetime
stores_df['Open Date'] = pd.to_datetime(stores_df['Open Date'], errors='coerce')

# Verify the conversion
print(stores_df['Open Date'].dtype)


In [ ]:
# Check for duplicates
duplicates = stores_df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Remove duplicates if necessary
if duplicates > 0:
    stores_df.drop_duplicates(inplace=True)


In [ ]:
# Summary statistics
print(stores_df.describe(include='all'))


In [ ]:
# Unique entries in 'Country'
print("Unique Countries:")
print(stores_df['Country'].value_counts())

# Unique entries in 'State'
print("\nUnique States:")
print(stores_df['State'].value_counts())


In [ ]:
# Load the sales data
sales_df = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Sales.csv", encoding='ISO-8859-1')

# Merge stores with sales data
merged_data = pd.merge(stores_df, sales_df, left_on='StoreKey', right_on='StoreKey', how='left')

# Check the merged DataFrame
print(merged_data.info())
print(merged_data.isnull().sum())

In [ ]:
# Handle missing values in the merged dataset

# Option 1: Fill missing values with placeholders
merged_data['Order Number'].fillna('Unknown', inplace=True)
merged_data['Line Item'].fillna(0, inplace=True)
merged_data['Order Date'].fillna(pd.to_datetime('1970-01-01'), inplace=True)
merged_data['CustomerKey'].fillna('Unknown', inplace=True)
merged_data['ProductKey'].fillna('Unknown', inplace=True)
merged_data['Quantity'].fillna(0, inplace=True)
merged_data['Currency Code'].fillna('USD', inplace=True)

# Option 2: Handle Delivery Date
# Given the significant number of missing values, you can drop this column or fill it
# Option to drop
merged_data.drop(columns=['Delivery Date'], inplace=True)

# Option to fill with NaT if you want to keep it
# merged_data['Delivery Date'].fillna(pd.NaT, inplace=True)

# Verify missing values again
print(merged_data.isnull().sum())

# Summary statistics after handling missing values
print(merged_data.describe(include='all'))


In [ ]:
# Assuming 'Sales Amount' is a column in merged_data
store_sales_performance = merged_data.groupby('StoreKey')['Quantity'].sum().reset_index()

# Sort by sales quantity
store_sales_performance = store_sales_performance.sort_values(by='Quantity', ascending=False)

print("Sales Performance by Store:")
print(store_sales_performance)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizing sales performance by store
plt.figure(figsize=(12, 6))
sns.barplot(x='Quantity', y='StoreKey', data=store_sales_performance.head(10))
plt.title('Top 10 Stores by Sales Quantity')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Store Key')
plt.show()


In [ ]:
# Correlation matrix
correlation_matrix = merged_data[['Square Meters', 'Quantity']].corr()

# Visualizing the correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


In [ ]:
# 1. Store Performance

# Calculate operational years
merged_data['Open Date'] = pd.to_datetime(merged_data['Open Date'])  # Ensure Open Date is in datetime format
merged_data['Operational Years'] = (pd.to_datetime('today') - merged_data['Open Date']).dt.days / 365

# Aggregate store performance metrics
store_performance = merged_data.groupby('StoreKey').agg({
    'Quantity': 'sum',
    'Square Meters': 'mean',
    'Operational Years': 'mean'
}).reset_index()

store_performance.columns = ['StoreKey', 'Total Sales Quantity', 'Average Size (m?)', 'Operational Years']
print("Store Performance Metrics:")
print(store_performance)


In [ ]:
# Calculate total sales quantity by country
geographical_sales = merged_data.groupby('Country')['Quantity'].sum().reset_index()
geographical_sales = geographical_sales.sort_values(by='Quantity', ascending=False)

print("Total Sales by Country:")
print(geographical_sales)

# If you want to analyze by State as well
state_sales = merged_data.groupby('State')['Quantity'].sum().reset_index()
state_sales = state_sales.sort_values(by='Quantity', ascending=False)

print("Total Sales by State:")
print(state_sales)


In [ ]:
# Visualizing sales by country
plt.figure(figsize=(12, 6))
sns.barplot(x='Quantity', y='Country', data=geographical_sales)
plt.title('Total Sales Quantity by Country')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Country')
plt.show()


In [ ]:
# Visualizing sales by state
plt.figure(figsize=(12, 6))
sns.barplot(x='Quantity', y='State', data=state_sales)
plt.title('Total Sales Quantity by State')
plt.xlabel('Total Quantity Sold')
plt.ylabel('State')
plt.show()


In [ ]:
import pandas as pd

# Define the path for saving the combined CSV file
output_path = r"D:\PROJECTS\EDA\Cleanned Data\store_analysis.csv"

# Create a dictionary of your DataFrames
dataframes = {
    'Stores Data': stores_df,
    'Store_analysis Data': merged_data,
    'Store Performance': store_performance,
    'Sales by Country': geographical_sales,
    'Sales by State': state_sales
}

# Concatenate all DataFrames into one using a different variable name
all_data = pd.concat(
    [df.assign(Source=sheet_name) for sheet_name, df in dataframes.items()],
    ignore_index=True
)

# Save the combined DataFrame to a single CSV file
all_data.to_csv(output_path, index=False)

print(f"Data successfully saved to {output_path}")


In [ ]:
all_data

In [ ]:
all_data.dtypes

In [ ]:
from sqlalchemy import create_engine

# Database connection details
user = 'root'
password = 'Krishna3_6_9'
host = '127.0.0.1'
database = 'Datas'

# Create a database connection
connection_string = f'mysql+pymysql://{user}:{password}@{host}/{database}'
engine = create_engine(connection_string)

# Store the purchase_summary DataFrame to the database
all_data.to_sql('Store_analysis', con=engine, index=False, if_exists='replace')

print("Data successfully stored in the MySQL database.")


In [ ]:
store_performance.dtypes

In [ ]:
geographical_sales.dtypes

In [ ]:
state_sales.dtypes

In [ ]:
state_sales.dtypes